# 讲课笔记：Day 1 下午 — Transformer 架构（3.5h）

**讲师用讲课指南** — 本笔记对应 `Day1_下午_Transformer架构.ipynb`

---

## 整体时间安排

| 时段 | 内容 | 时长 | 关键目标 |
|------|------|------|----------|
| 0:00-0:10 | Part 1: 回顾静态向量的局限 | 10min | 建立"为什么需要Attention"的动机 |
| 0:10-0:50 | Part 2: Self-Attention QKV 机制 | 40min | 手把手拆解公式+代码+练习1 |
| 0:50-1:15 | Part 3: Multi-Head Attention | 25min | 理解多头分工+练习2 |
| 1:15-1:25 | 休息 | 10min | |
| 1:25-2:00 | Part 4: Transformer Block | 35min | 残差+LN+FFN+组装+练习3 |
| 2:00-2:40 | Part 5: GPT 组装与生成 | 40min | 完整GPT+采样策略+练习4 |
| 2:40-3:10 | Mini-project + 练习5 | 30min | 企业场景+参数量估算 |
| 3:10-3:30 | Part 6: 架构对比 + 总结 | 20min | GPT/BERT/T5 + Day 2 预告 |

**讲课节奏提醒**：
- 代码随讲部分：讲师屏幕共享 notebook，一个 cell 一个 cell 地跑
- 每个 cell 先讲概念，再运行，再解读输出
- 练习时间要严格控制，提前给提示避免卡壳

---

# 开场准备（上课前 5 分钟）

## 📍 Cell 1-3（setup cells）

⏱ **时间**：上课前静默运行，不需要讲解

🎯 **操作**：
- Cell 1: `sys.path` 配置 — 直接运行
- Cell 2: 中文字体配置 — 直接运行，确认输出 `已加载中文字体: Noto Sans CJK SC`
- Cell 3: PyTorch import — 直接运行，确认输出 `PyTorch version: 2.x.x`

**如果出错**：
- 字体找不到：检查 `../fonts/` 路径是否正确
- PyTorch 版本不对：确认虚拟环境激活正确

**讲师自查**：确保所有 cell 能跑通，热力图中文不乱码

---

# Part 1 | 回顾：为什么静态向量不够？（10 min）

## 📍 Cell 4（Markdown cell — Part 1 标题+说明）

⏱ **时间**：0:00-0:10（10分钟）

🎯 **核心概念**：从上午的 Word Embedding 过渡到 Self-Attention 的动机

🗣 **讲课话术**：

> 好，同学们，下午好！上午我们学了词嵌入对吧？把每个词变成一个向量。但你们有没有想过一个问题——同一个词在不同句子里意思完全不一样怎么办？
>
> 比如说"银行"这个词——"我去银行存钱"，这里银行是金融机构；"河边有一排柳树"，英文里 bank 还有河岸的意思。但是在静态 Embedding 里，bank 永远只有一个向量，不管上下文是什么，给你的都是同一个表示。
>
> 这就是上午那套方法的根本局限——**静态的**。一个词一个向量，死的，不会变。
>
> 那怎么办？我们需要一种机制，让每个词能"看看"周围的词，然后根据上下文来动态调整自己的表示。这就是 **Self-Attention（自注意力）** 的核心思想。
>
> 打个比方：你在一个会议室里，你是"银行"这个词。你看看左边的人说"存钱"，右边的人说"利息"——哦，原来我是金融机构。换个场景，你左边是"河流"，右边是"柳树"——那我就是河岸。**Self-Attention 就是这个"看看周围人"的过程。**

## 📍 Cell 5（Markdown cell — QKV 直觉理解）

🗣 **讲课话术**：

> 在正式看公式之前，我先给你们一个直觉。想象你在图书馆找书：
>
> - **Query（查询）** = 你脑中的问题："我想找关于深度学习的书"
> - **Key（键）** = 每本书封面上的标签："机器学习"、"烹饪"、"历史"
> - **Value（值）** = 书里面的实际内容
>
> Attention 的过程就是：你拿着你的问题（Q），去和每本书的标签（K）比对，找到最匹配的那几本，然后提取它们的内容（V）。匹配度越高的书，你从中提取的信息就越多。
>
> 在 Self-Attention 里，每个词同时扮演三个角色——它既会"提问题"，也会"亮出自己的标签"，还会"准备好自己的内容"。酷不酷？记住这个图书馆的比喻，后面看公式就容易理解了。

❓ **常见学生问题**：

- **Q: 为什么不直接用词向量做 attention，还要分 Q/K/V？**
  - A: 因为"你要找什么"和"你能提供什么"可能是不同的。比如"猫"在做查询时可能在找动作（谁坐了？），但作为内容提供者它提供的是"毛茸茸的动物"这个信息。分开三个角色让模型更灵活。

- **Q: Self-Attention 和普通 Attention 有什么区别？**
  - A: Self 的意思是 Q、K、V 都来自同一个序列。在机器翻译里，Q 来自解码器、K/V 来自编码器，那就是 Cross-Attention。

➡️ **过渡**：好，直觉有了，现在我们来看具体的数学公式和代码实现。

---

# Part 2 | Self-Attention：QKV 机制（40 min）

## 📍 Cell 6（Markdown cell — 核心公式）

⏱ **时间**：0:10-0:15（5分钟讲公式）

🎯 **核心概念**：Attention 公式 + 为什么要缩放

🗣 **讲课话术**：

> 好，公式来了。别怕，其实就四步：
>
> $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$
>
> 翻译成人话就是：
> 1. **QK^T** — 用你的问题去和所有标签做点积，得到"匹配分数"
> 2. **除以 sqrt(d_k)** — 缩放一下，防止分数太大
> 3. **softmax** — 把分数变成概率（加起来等于1）
> 4. **乘以 V** — 按概率加权提取内容
>
> 那个 **除以 sqrt(d_k)** 是一个很巧妙的设计。为什么要除？想象一下，如果维度 d_k 是 512，两个向量点积出来的值可能是好几百。softmax 一算，概率分布就变成了接近 one-hot 的东西——0.999 和 0.001。梯度就没了，模型学不动。除以 sqrt(512) 约等于 22.6，把值缩小到合理范围，softmax 输出更平滑，梯度更稳定。
>
> 这个技巧叫 **scaled dot-product attention**，是 2017 年 "Attention is All You Need" 论文里提出的。

👀 **板书/PPT 要点**：
- 在白板上画出 Q、K、V 三个矩阵的维度
- 强调 QK^T 的结果是 [seq_len, seq_len] 的方阵

❓ **常见问题**：
- **Q: 为什么是 sqrt 而不是直接除以 d_k？**
  - A: 点积的方差和维度成正比（假设各维独立且方差为1，点积方差 = d_k），所以除以 sqrt(d_k) 让方差回到 1。这是统计学上的标准化。

## 📍 Cell 7（Markdown — Step-by-step 标题）+ Cell 8（代码：创建输入 x）

⏱ **时间**：0:15-0:18（3分钟）

🎯 **核心概念**：用"猫坐垫"3个词、4维向量的迷你例子来手把手演示

🗣 **讲课话术**：

> 好，公式看着抽象，我们来用一个超级小的例子走一遍。
>
> 假设我们有 3 个词："猫"、"坐"、"垫"，每个词用 4 维向量表示。（实际 GPT 用 768 甚至 12288 维，但 4 维方便我们看数字）
>
> 【运行 Cell 8】

👀 **指出输出中的关键信息**：
- `形状: [3, 4] = [词数, 维度]` — 3 行对应 3 个词，4 列对应 4 个维度
- 每一行就是一个词的 embedding 向量
- 数字是随机初始化的（`torch.manual_seed(42)`），所以每次运行结果一样

> 现在 x 矩阵有了，每行一个词。接下来我们要用三个权重矩阵 W_q、W_k、W_v 把 x 变成 Q、K、V。

## 📍 Cell 9（代码：线性投影得到 Q, K, V）

⏱ **时间**：0:18-0:22（4分钟）

🎯 **核心概念**：W_q, W_k, W_v 线性投影，把同一个 x 变成三种不同的表示

🗣 **讲课话术**：

> 看这个 cell。我们创建了三个 `nn.Linear` 层：W_q、W_k、W_v。
>
> 它们干的事情就是——同一个输入 x，乘以三个不同的权重矩阵，得到三个不同的向量。
> - Q = x @ W_q — "这个词在找什么？"
> - K = x @ W_k — "这个词能提供什么标签？"
> - V = x @ W_v — "这个词实际提供什么内容？"
>
> 【运行 Cell 9】
>
> 你看输出，Q、K、V 都是 [3, 4] 的矩阵，和输入 x 形状一样。但数值不同了——因为乘了不同的权重矩阵。
>
> 回到图书馆的比喻：W_q 把你的需求翻译成搜索关键词，W_k 把每本书的内容翻译成标签，W_v 把每本书的内容提炼成摘要。这三个翻译器是模型学习出来的！

👀 **指出输出**：
- Q、K、V 三个矩阵都打印出来了
- 形状都是 [3, 4]：3 个词，4 个维度
- 强调这些权重矩阵是"可学习的"，训练过程中会不断调整

❓ **常见问题**：
- **Q: bias=False 是什么意思？**
  - A: 线性层默认有偏置项 y = Wx + b，这里设 bias=False 就没有 b，只有 y = Wx。很多 Attention 实现里偏置是可选的，GPT-2 用了偏置，LLaMA 没用。

## 📍 Cell 10（代码：计算注意力分数 scores = QK^T）

⏱ **时间**：0:22-0:25（3分钟）

🎯 **核心概念**：QK^T 矩阵乘法得到注意力分数矩阵

🗣 **讲课话术**：

> 现在到了最关键的一步——算 QK^T。
>
> Q 是 [3, 4]，K 转置后是 [4, 3]，矩阵乘法得到 [3, 3]。
>
> 这个 3x3 矩阵的含义是什么？第 i 行第 j 列的值，就是"第 i 个词对第 j 个词的关注程度"。
>
> 比如 scores[0][2] 就是"猫"对"垫"的关注分数。分数越高，说明"猫"越关注"垫"。
>
> 然后除以 sqrt(d_k) = sqrt(4) = 2，做缩放。
>
> 【运行 Cell 10】

👀 **指出输出中的关键信息**：
- scores 矩阵的形状是 [3, 3]——3个词对3个词
- 看看哪些分数特别高或特别低
- 缩放后的值应该在合理范围内（大约 -2 到 2）

## 📍 Cell 11（代码：Softmax 得到注意力权重）

⏱ **时间**：0:25-0:27（2分钟）

🎯 **核心概念**：Softmax 把分数转成概率分布

🗣 **讲课话术**：

> 现在 softmax 登场。它把每一行的分数变成概率——加起来等于 1。
>
> 为什么要变成概率？因为我们接下来要做"加权平均"。权重必须加起来等于 1 才有意义。
>
> 【运行 Cell 11】

👀 **指出输出**：
- 每一行的数字加起来应该是 1.0
- 看"猫"这一行，它最关注哪个词？是自己还是"坐"或"垫"？
- 注意力权重告诉我们：模型认为这些词之间的关系是什么样的

## 📍 Cell 12（代码：加权求和得到输出）

⏱ **时间**：0:27-0:29（2分钟）

🎯 **核心概念**：attention_weights @ V = 加权混合后的新表示

🗣 **讲课话术**：

> 最后一步，用注意力权重去加权 V。
>
> 比如"猫"这个词，它的新表示 = 0.3 * V_猫 + 0.5 * V_坐 + 0.2 * V_垫（举例数字，以实际输出为准）
>
> 这意味着"猫"的新表示融合了所有词的信息，但重点关注它最"在意"的那些词。这就是"上下文感知"的表示！
>
> 输出还是 [3, 4]——形状没变，但内容变了。每个词的向量现在包含了上下文信息。
>
> 对比一下原始的 x 和输出：同样是 4 维向量，但输出的向量是"看过"了其他词之后的结果。这就是从"静态 Embedding"到"动态 Embedding"的飞跃！
>
> 【运行 Cell 12】

👀 **指出输出**：
- 输出形状 [3, 4] — 和输入一样
- 但数值不同了 — 每个词现在"知道"了周围的词

## 📍 Cell 13（代码：注意力权重热力图）

⏱ **时间**：0:29-0:33（4分钟）

🎯 **核心概念**：可视化注意力权重，直观理解"谁在关注谁"

🗣 **讲课话术**：

> 数字看多了头疼，我们来画个热力图！
>
> 【运行 Cell 13】
>
> 看这个热力图。横轴是 Key（被关注的词），纵轴是 Query（发起关注的词）。颜色越深，关注度越高。
>
> 比如你看"坐"那一行，它最关注谁？可能最关注"猫"——因为"坐"需要知道"谁在坐"。
>
> 再看"垫"那一行——它可能关注"坐"，因为"坐在什么上面"。
>
> 当然，这里的权重是随机初始化的，没有经过训练，所以模式可能不太明显。但训练之后，这些权重会学到真正有意义的关系模式。
>
> 大家记住这个热力图的形式，后面我们还会反复看到它。在研究大模型行为时，attention 热力图是最常用的分析工具之一。

👀 **指出图中的关键点**：
- 热力图的 x 轴和 y 轴分别标注了"猫"、"坐"、"垫"
- 对角线上的值通常不低——每个词都会一定程度关注自己
- 数值标注在每个格子里（`annot=True, fmt='.3f'`）

❓ **常见问题**：
- **Q: 为什么很多词更关注自己？**
  - A: 这在未训练的模型里是随机的。在训练好的模型里，有些 head 确实会学到"关注自己"的模式（identity head），有些会学到"关注前一个词"的模式。不同 head 分工不同，后面 Multi-Head 部分会讲到。

## 📍 Cell 14-15（完整的 Self-Attention 函数 + 测试）

⏱ **时间**：0:33-0:36（3分钟）

🎯 **核心概念**：把前面的步骤封装成一个函数，加入 batch 维度

🗣 **讲课话术**：

> 好，前面我们一步步拆开看了 Self-Attention。现在把它打包成一个函数。
>
> 注意这个版本是"最简单的"——Q=K=V=x，没有投影矩阵。就是直接用输入做 attention。
>
> 【运行 Cell 15】
>
> 看输出形状：
> - 输入 [1, 5, 8]：1个batch，5个词，8维
> - 输出 [1, 5, 8]：形状一样
> - 注意力权重 [1, 5, 5]：5个词对5个词的关注矩阵

👀 **指出代码结构**：
- 函数签名清晰：`self_attention(x, d_k)` 返回 `(output, attention_weights)`
- 四步对应公式的四步
- `K.transpose(-2, -1)` 是在最后两个维度上转置

## 📍 Cell 16-17（因果掩码 Causal Mask）

⏱ **时间**：0:36-0:40（4分钟）

🎯 **核心概念**：生成式模型（GPT）只能看到过去的词，不能看到未来

🗣 **讲课话术**：

> 等一下，我们刚才的 attention 有个问题——每个词能看到所有词，包括它后面的词。
>
> 这对 BERT 这种理解模型来说没问题，但对 GPT 这种生成模型不行。为什么？
>
> 因为 GPT 是"预测下一个词"的。如果你在预测第 3 个词的时候就能看到第 4、5 个词，那不就是作弊吗？考试的时候偷看答案，模型永远学不会真正的预测能力。
>
> 所以我们需要一个"因果掩码"（Causal Mask）——一个下三角矩阵。
>
> 【运行 Cell 16/17 — 因果掩码可视化】
>
> 看这个图：
> - 1 表示"可以看到"，0 表示"不能看到"
> - 第 0 个位置只能看到自己
> - 第 1 个位置能看到自己和位置 0
> - 第 5 个位置能看到 0-5 所有位置
>
> 实现上很简单：`torch.tril(torch.ones(n, n))`，然后把 0 的位置在 scores 里设成负无穷，softmax 后就变成 0。
>
> 这就是 GPT 和 BERT 的最大区别之一——GPT 用因果掩码（单向），BERT 不用（双向）。

👀 **指出图中的关键信息**：
- 下三角矩阵的形状
- `1 = 可以关注, 0 = 不可关注`
- 对角线上全是 1——每个位置至少能看到自己

❓ **常见问题**：
- **Q: 为什么把被遮蔽的位置设成 -inf 而不是 0？**
  - A: 因为我们要在 softmax 之前设置。softmax(-inf) = 0，这样被遮蔽的位置对最终输出完全没有贡献。如果设成 0，softmax(0) 是一个正数，还是会有影响。

➡️ **过渡**：好，Self-Attention 的核心我们讲完了。现在来做第一个练习，检验一下大家理解了没有。

## 📍 Cell 18-19（练习 1：手算 Attention 分数）

⏱ **时间**：0:40-0:50（10分钟练习）

🎯 **核心概念**：用简化数值手算 Self-Attention 全过程

🗣 **练习引导**：

> 好，现在轮到你们动手了。这个练习很简单——用计算器就能做。
>
> 2 个词，每个词 64 维，所有维度值相同。x1 全是 1.323，x2 全是 1.134。Q=K=V=x，没有投影矩阵。
>
> 你们需要算 4 步：
> 1. **点积**：q2·k1 = 1.134 * 1.323 * 64 = ?（每个维度的乘积加起来）
> 2. **缩放**：除以 sqrt(64) = 8
> 3. **Softmax**：两个缩放分数做 softmax
> 4. **加权求和**：z2 = w1 * x1 + w2 * x2
>
> 先自己算，然后运行 cell 看答案。给你们 5 分钟。

**预期输出**：
```
Step 1: q2·k1 = 96.27, q2·k2 = 82.31
Step 2: scaled scores = [12.03, 10.29]
Step 3: attention weights = [0.851, 0.149]
Step 4: z2 = 0.851 * 1.323 + 0.149 * 1.134 = 1.2949
```

🗣 **讲解要点**：

> 看结果——x2 的新表示 z2 = 1.2949，比 x2 原来的 1.134 大，比 x1 的 1.323 小。为什么？因为 x1 的值更大，点积更大，attention 权重更高（0.851 vs 0.149），所以 z2 被"拉向"了 x1。
>
> 这就是 Attention 的本质——**信息的加权混合**。每个词的输出都是所有词的加权平均，权重由 QK 的匹配度决定。

**如果学生卡壳**：
- 提示1：点积 = 逐元素相乘再求和。所有维度值相同时，q2·k1 = 1.134 * 1.323 * 64
- 提示2：softmax(a, b) = [exp(a)/(exp(a)+exp(b)), exp(b)/(exp(a)+exp(b))]
- 提示3：加权求和就是 w[0]*x1 + w[1]*x2

➡️ **过渡**：好，一个 head 的 attention 我们搞定了。但一个 head 只能学一种模式——有的词对关注语法关系，有的关注语义。怎么同时学多种模式？答案是 Multi-Head Attention。

---

# Part 3 | Multi-Head Attention（25 min）

## 📍 Cell 20（Markdown — Part 3 标题+说明）

⏱ **时间**：0:50-0:55（5分钟讲概念）

🎯 **核心概念**：多头注意力的动机和实现方式

🗣 **讲课话术**：

> 刚才我们只有一个 Attention Head，它只能学一种"关注模式"。
>
> 但语言的关系是多层次的：
> - 语法关系：主语和谓语要匹配（"猫"和"坐"）
> - 语义关系：同义词和近义词
> - 位置关系：相邻的词
> - 长距离依赖：代词和它指代的名词
>
> 一个 head 搞不定这么多关系。怎么办？**多请几个专家！**
>
> Multi-Head Attention 的做法：
> 1. 把 d_model 维度拆成 n_heads 份（比如 768 拆成 12 份，每份 64 维）
> 2. 每个 head 在自己的 64 维子空间里独立做 attention
> 3. 最后把所有 head 的结果拼起来，再过一个线性层
>
> 打个比方：你要了解一个人，你可以同时问 8 个不同角度的问题——外貌、性格、专业能力、兴趣爱好……每个问题得到不同的信息，最后综合起来就是全面的了解。每个 head 就像一个提问角度。

## 📍 Cell 21（代码：MultiHeadAttention 类）

⏱ **时间**：0:55-1:00（5分钟讲代码）

🎯 **核心概念**：多头注意力的代码实现

🗣 **讲课话术**：

> 看代码。关键操作：
>
> 1. **投影**：还是 W_q、W_k、W_v，但输出维度是完整的 d_model
> 2. **Reshape**：`.view(B, T, n_heads, d_k).transpose(1, 2)` — 这一步把 [B, T, d_model] 变成 [B, n_heads, T, d_k]
> 3. **独立 Attention**：每个 head 在自己的 d_k 维空间里做 attention
> 4. **拼回来**：`.transpose(1, 2).contiguous().view(B, T, d_model)`
> 5. **输出投影**：W_o 把拼接结果再投影一次
>
> 关键理解：**多头不是多次计算，而是一次矩阵运算里并行完成的**。GPU 上效率很高。
>
> 【运行 Cell 21】

👀 **指出输出**：
- 输入 `[2, 10, 64]`：batch=2，10个词，64维
- 输出 `[2, 10, 64]`：形状完全不变
- 注意力权重 `[2, 8, 10, 10]`：**8个头**，每个头都有 10x10 的注意力矩阵

> 看到了吗？注意力权重从 [10, 10] 变成了 [8, 10, 10]——8 个 head，每个都有自己的注意力模式。

## 📍 Cell 22（代码：可视化不同 Head 的注意力模式）

⏱ **时间**：1:00-1:05（5分钟）

🎯 **核心概念**：直观看到不同 head 学到不同的关注模式

🗣 **讲课话术**：

> 现在我们来看最有意思的部分——不同 head 关注了什么？
>
> 我用一句话"我喜欢用大模型写代码"跑了一下 4 头的 attention。
>
> 【运行 Cell 22】
>
> 看这 4 个热力图！
> - Head 0：可能比较均匀地关注所有词
> - Head 1：可能更关注相邻的词
> - Head 2：可能有一些特别亮的格子——集中关注某个词
> - Head 3：又是另一种模式
>
> 每个 head 真的学到了不同的"观察角度"！这就是多头注意力的威力。
>
> 当然这里模型没有经过训练，模式是随机的。在真正训练好的大模型里，研究者发现有些 head 专门负责追踪语法（比如主谓一致），有些负责追踪代词指代，有些负责关注标点符号。非常有意思。

👀 **指出图中的关键点**：
- 4 个子图排列，标题 `Head 0` 到 `Head 3`
- 每个热力图的模式不同——有的均匀，有的集中
- x/y 轴标注了中文词："我"、"喜欢"、"用"、"大"、"模型"、"写代码"

## 📍 Cell 23-24（练习 2：注意力模式侦探）

⏱ **时间**：1:05-1:15（10分钟练习）

🎯 **核心概念**：用熵（entropy）量化分析不同 head 的关注模式

🗣 **练习引导**：

> 刚才我们用眼睛看热力图，现在我们用数学来量化——用**熵**。
>
> 熵是什么？信息论里的概念：
> - **高熵** = 分布很均匀，不确定性高 → 广泛关注，"雨露均沾"
> - **低熵** = 分布很集中，不确定性低 → 集中关注，"情有独钟"
>
> 公式：H = -sum(w * log(w + eps))
>
> 你们需要：
> 1. 用 mha 的权重矩阵提取每个 head 的注意力权重（Q、K 投影 + reshape）
> 2. 计算每个 head 的平均熵
> 3. 找出最"集中"的 head（熵最低的）
>
> 5 分钟自己试，看 hints 也可以。

**预期输出**：
```
Head |    输入A 熵 |    输入B 熵 |     差异
----------------------------------------
   0 |   1.7420 |   1.3428 | 0.3992
   1 |   1.7299 |   1.4317 | 0.2982
   2 |   1.7504 |   1.0300 | 0.7205
   3 |   1.7422 |   1.3702 | 0.3720

最集中的Head: 1（熵=1.7299）
```

🗣 **讲解要点**：

> 看结果——Head 1 的熵最低（1.7299），说明它的注意力分布最集中，最"专注"。
>
> 再看差异列——Head 2 在两个不同输入上的熵差异最大（0.7205），说明这个 head 对输入内容最敏感。而 Head 1 差异最小，说明它不管输入是什么都保持类似的关注模式——可能学到了某种位置性的规律。
>
> 这在大模型研究里叫 **head analysis**（注意力头分析），是理解模型行为的重要工具。

❓ **常见问题**：
- **Q: eps 是什么？**
  - A: 一个很小的数（如 1e-8），防止 log(0) 报错。因为如果某个注意力权重恰好是 0，log(0) = -inf。

➡️ **过渡**：好，休息 10 分钟！回来我们讲 Transformer Block——把 attention 和其他组件组装在一起。

---

# 休息 10 分钟（1:15-1:25）

## 📍 Cell 25（Markdown — 休息提示）

🗣 **休息前说一句**：

> 休息 10 分钟。到目前为止我们已经搞定了 Self-Attention 的核心——QKV 机制和 Multi-Head。下半场我们要把它变成一个完整的 Transformer Block，然后组装成 GPT。节奏会加快一点，但概念都是基于前面的基础。

**讲师趁休息时间做的事**：
- 检查后续 cell 是否能正常运行
- 看看聊天区有没有遗留的问题
- 喝口水

---

# Part 4 | Transformer Block（35 min）

## 📍 Cell 26（Markdown — Part 4 标题+架构图）

⏱ **时间**：1:25-1:30（5分钟概述）

🎯 **核心概念**：Transformer Block 的三个关键组件

🗣 **讲课话术**：

> 好，回来了。下半场的第一个目标——搭建完整的 Transformer Block。
>
> 如果把 Multi-Head Attention 比作"大脑"，那 Transformer Block 就是"完整的神经系统"。光有大脑不够，还需要：
> 1. **残差连接（Residual Connection）** — 高速公路，让梯度直达
> 2. **层归一化（Layer Normalization）** — 稳压器，稳定训练
> 3. **前馈网络（FFN）** — 加工厂，增加非线性变换能力
>
> 看看 notebook 里的 ASCII 架构图：
> ```
> x → LayerNorm → Attention → (+) → LayerNorm → FFN → (+)
>     |_________________________|    |__________________|  
>            残差连接                      残差连接
> ```
>
> 这叫 **Pre-LN**（先归一化再变换），GPT-2 以后的模型基本都用这种结构。原始 Transformer 用的是 Post-LN（先变换再归一化），训练不太稳定。

**白板画图要点**：
- 画出数据流：x 进来，分两路（一路过 LN+Attn，一路直连），在加号处汇合
- 强调残差连接是"跳过"整个 attention/FFN 的捷径

## 📍 Cell 27（Markdown — 残差连接说明）+ Cell 28（代码：残差连接对比实验）

⏱ **时间**：1:30-1:35（5分钟）

🎯 **核心概念**：残差连接解决深层网络梯度消失

🗣 **讲课话术**：

> 残差连接的核心就一行代码：`output = f(x) + x`
>
> 这一行有多重要？没有它，超过 10 层的网络基本训不动。为什么？
>
> 数学上来看：反向传播时，梯度要逐层回传。如果每层都是 `f(x)`，梯度 = df1/dx * df2/dx * ... * df10/dx，连乘 10 次，要么爆炸要么消失。
>
> 加了残差之后：`f(x) + x`，梯度 = df/dx + 1。那个 **+ 1** 就是救命稻草——不管 df/dx 多小，梯度至少还有 1，不会消失。
>
> 打个比方：你要爬 100 层楼，每层都要过一个安检（f(x)）。没残差连接就像每个安检都可能把你拦住；有残差连接就像每层都有个电梯直达——就算安检把你拦了，你还能走电梯。
>
> 【运行 Cell 28】

👀 **指出输出中的关键对比**：
- `无残差 - 输出范围: [-0.21, 0.25]` — 信号几乎被压没了！
- `有残差 - 输出范围: [-14.95, 14.20]` — 信号保持得很好
- 10 层之后，差距已经这么大。GPT-3 有 96 层，没残差根本不可能训练

❓ **常见问题**：
- **Q: 残差连接是 ResNet 提出的吗？**
  - A: 是的！何恺明 2015 年在 ResNet 论文里提出。Transformer 2017 年借鉴了这个思想。可以说残差连接是深度学习最重要的技术之一。

## 📍 Cell 29（Markdown — LayerNorm 公式）+ Cell 30（代码：手写 LayerNorm）

⏱ **时间**：1:35-1:40（5分钟）

🎯 **核心概念**：LayerNorm 的原理和手写实现

🗣 **讲课话术**：

> 接下来是 LayerNorm。公式很简单：
>
> $$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$
>
> 翻译成人话：
> 1. 算均值 μ 和方差 σ^2（在特征维度上）
> 2. 减均值、除标准差 → 归一化到均值=0、方差=1
> 3. 乘以可学习的 γ 加上可学习的 β → 仿射变换
>
> 为什么需要它？因为每一层的输出数值范围可能越来越大或越来越小，训练会不稳定。LayerNorm 把每一层的输出拉回到标准范围。
>
> 注意这里和 BatchNorm 不同：
> - BatchNorm：在 batch 维度上归一化（不同样本同一特征）
> - LayerNorm：在特征维度上归一化（同一样本不同特征）
> - Transformer 用 LayerNorm 因为序列长度可变，BatchNorm 不好用
>
> 让我们来手写一个看看。
>
> 【运行 Cell 30】

👀 **指出输出中的关键数字**：
- 归一化前：均值约 5.x，方差约 90-110 → 很不标准
- 归一化后：均值接近 0（约 -2e-08），方差接近 1（约 1.016）→ 非常标准
- **和 PyTorch 官方实现的差距: 0.00000048** → 几乎一模一样！

🗣 **重点强调**：

> 看到了吗？差距是 4.8e-7，小数点后第 7 位才有差异。这说明我们手写的和 PyTorch 官方的 `nn.LayerNorm` 是等价的。以后用 PyTorch 提供的就行，但理解原理很重要。

❓ **常见问题**：
- **Q: ε 是什么？**
  - A: 一个很小的数（默认 1e-5），防止除以 0。如果某一层所有值都相同，方差就是 0，没有 ε 就会报错。
- **Q: γ 和 β 初始化为什么？**
  - A: γ 初始化为全 1，β 初始化为全 0。也就是说一开始 LayerNorm 基本就是标准归一化，训练过程中 γ 和 β 会调整，让模型可以学到每个维度最合适的尺度和偏移。

## 📍 Cell 31（Markdown — FFN 说明）+ Cell 32（代码：FeedForward 类）

⏱ **时间**：1:40-1:45（5分钟）

🎯 **核心概念**：前馈网络 = 两层线性变换 + GELU 激活，4x 扩展

🗣 **讲课话术**：

> Transformer Block 里的第二个大组件——前馈网络（FFN）。
>
> 结构超级简单：Linear → GELU → Linear。就是两层全连接。
>
> 但有个关键细节——**中间层扩展 4 倍**！如果 d_model=64，中间层是 256。先把信息"展开"到更高维空间做非线性变换，再压缩回来。
>
> 为什么要 4 倍？原始论文就是 4 倍，后来的实验也验证了这是个很好的经验值。太小学不到足够复杂的模式，太大参数浪费。
>
> 有个很重要的知识点：**FFN 占了 Transformer 总参数量的 60-70%**！ 对，你没听错。大家以为 Attention 是 Transformer 的核心，但从参数量来说，FFN 才是"大户"。
>
> 为什么？因为 FFN 的两个线性层分别是 [d_model, 4*d_model] 和 [4*d_model, d_model]，总共 8*d_model^2 的参数。而 Attention 的 Q/K/V/O 四个投影总共 4*d_model^2。FFN 是 Attention 的两倍！
>
> 【运行 Cell 32】

👀 **指出输出**：
- 输入 `[2, 10, 64]`，中间层 `[2, 10, 256]`（4x 扩展），输出 `[2, 10, 64]`
- 参数量 33,088 — 后面和 attention 对比

🗣 **补充说明 GELU**：

> GELU 是什么？全名 Gaussian Error Linear Unit。和 ReLU 类似但更平滑——ReLU 在 0 处有尖角，GELU 是平滑过渡的。GPT-2/3/4、BERT、LLaMA 都用 GELU。你们可以把它理解成"更高级的 ReLU"。

## 📍 Cell 33（Markdown — 组装标题）+ Cell 34（代码：MultiHeadAttentionV2）+ Cell 35（代码：TransformerBlock 类）

⏱ **时间**：1:45-1:50（5分钟）

🎯 **核心概念**：Pre-LN 架构，把所有组件装在一起

🗣 **讲课话术**：

> 好，积木都有了——Attention、LayerNorm、FFN、Residual。现在我们来拼装！
>
> Cell 34 是一个加了 dropout 的 MultiHeadAttention（V2 版本，更完整）。
>
> Cell 35 是完整的 TransformerBlock。看 forward 方法，就两行代码：
> ```python
> x = x + self.dropout(self.attention(self.ln1(x), mask))
> x = x + self.dropout(self.ffn(self.ln2(x)))
> ```
>
> 第一行：先 LayerNorm，再 Attention，再 Dropout，然后加上残差
> 第二行：先 LayerNorm，再 FFN，再 Dropout，然后加上残差
>
> 就这么简单！这就是一个 Transformer Block 的全部。GPT-2 里有 12 个这样的 block 堆叠起来，GPT-3 有 96 个。
>
> 【运行 Cell 34 和 35】

👀 **指出输出**：
- 输入 `[2, 10, 64]`，输出 `[2, 10, 64]` — 维度不变！
- 参数量 49,984 — 一个 block 就有 5 万参数
- 注意 forward 方法里的 `x + ...` 就是残差连接

❓ **常见问题**：
- **Q: Pre-LN 和 Post-LN 有什么区别？**
  - A: Pre-LN 是先归一化再做变换（x + Attn(LN(x))），Post-LN 是先变换再归一化（LN(x + Attn(x))）。Pre-LN 训练更稳定，不需要 warmup，GPT-2 以后基本都用 Pre-LN。
- **Q: dropout 在推理时还有用吗？**
  - A: 没有。推理时 `model.eval()` 会自动关掉 dropout。dropout 只在训练时起作用，用来防止过拟合。

## 📍 Cell 36-37（练习 3：从组件拼装 TransformerBlock）

⏱ **时间**：1:50-2:00（10分钟练习）

🎯 **核心概念**：自己动手组装，加深理解

🗣 **练习引导**：

> 好，现在轮到你们自己拼了。代码框架已经给你们了，你需要填两个地方：
> 1. `__init__` 里创建 5 个组件：ln1, ln2, attention, ffn, dropout
> 2. `forward` 里按 Pre-LN 结构连起来：两行代码
>
> 前面刚看了完整实现，现在凭记忆写一遍。5 分钟。
>
> 如果卡住了，看 hints：
> - 提示1：用 `nn.LayerNorm(d_model)` 创建 LN
> - 提示2：forward 的连接方式是 `x = x + self.dropout(self.attention(self.ln1(x), mask))`

**预期输出**：
```
输入: torch.Size([2, 10, 64])
输出: torch.Size([2, 10, 64])
参数量: 49,984
通过!
```

🗣 **讲解要点**：

> 参数量和前面的官方实现一样——49,984。说明你搭对了！
>
> 记住这个数字的构成：
> - Attention (Q/K/V/O): 4 * 64^2 + 4 * 64 = 16,640
> - FFN (两个线性层): 64*256 + 256 + 256*64 + 64 = 33,088
> - 两个 LayerNorm: 2 * (64 + 64) = 256
> - 总计: 16,640 + 33,088 + 256 = 49,984
>
> FFN 占了 33,088 / 49,984 = 66%！果然是大户。

➡️ **过渡**：好，一个 Transformer Block 搭好了。现在我们要把它变成一个完整的 GPT——加上 Embedding、位置编码、然后堆叠多层。

---

# Part 5 | GPT 组装与文本生成（40 min）

## 📍 Cell 38（代码：堆叠多个 Transformer Block）

⏱ **时间**：2:00-2:03（3分钟）

🎯 **核心概念**：多层堆叠 = 更深的理解

🗣 **讲课话术**：

> 一个 Transformer Block 处理一遍输入。多个 block 堆叠起来，每层都在上一层的基础上进一步理解。
>
> 类比：读一篇文章，第一遍只看到词面意思（第 1 层），第二遍理解句子关系（第 2-3 层），第三遍理解段落逻辑（第 4-6 层），第四遍理解全文主旨（更深层）。
>
> GPT-2 有 12 层，GPT-3 有 96 层，GPT-4 据传约 120 层。
>
> 【运行 Cell 38】

👀 **指出输出**：
- 6 层 Transformer，输入输出形状不变
- 总参数量约 30 万——每层 5 万，6 层就是 30 万多一点（加上最终的 LayerNorm）

## 📍 Cell 39-40（Markdown — GPT 完整结构 + 位置编码说明）

⏱ **时间**：2:03-2:08（5分钟讲概念）

🎯 **核心概念**：GPT 的完整架构 = Embedding + 位置编码 + Transformer堆叠 + LM Head

🗣 **讲课话术**：

> 现在来看 GPT 的完整结构。从 Token ID 到输出概率分布，一共经历这些步骤：
>
> 1. **Token Embedding (wte)**：把 token ID 变成向量。上午学的。
> 2. **Position Embedding (wpe)**：加上位置信息。为什么需要？
>
> 因为 Self-Attention 本身是"位置无关"的。"猫吃鱼"和"鱼吃猫"如果不加位置编码，attention 算出来的结果一模一样！这不对——语序很重要。
>
> GPT-2 的位置编码是**可学习的** Embedding——每个位置有一个向量，让模型自己学每个位置该是什么。最大长度 1024（后来 GPT-4 扩展到 128K）。
>
> 3. **Dropout**：防过拟合
> 4. **Transformer Block x N**：核心处理
> 5. **最终 LayerNorm (ln_f)**：最后再归一化一次
> 6. **Linear (lm_head)**：把 d_model 维映射到 vocab_size 维的 logits
>
> 有个精彩的设计——**权重共享 (weight tying)**：lm_head 的权重和 token embedding 共享！这意味着"Token ID → 向量"和"向量 → Token ID"用的是同一套参数，减少了大量参数量（vocab_size * d_model 的参数只需要存一份）。
>
> GPT-2、GPT-3、LLaMA 都用了这个技巧。

## 📍 Cell 41-42（代码：CausalSelfAttention + MLP + Block 类）

⏱ **时间**：2:08-2:12（4分钟快速过代码）

🎯 **核心概念**：GPT 风格的组件实现（带因果掩码）

🗣 **讲课话术**：

> 这几个 cell 定义了 GPT 专用的组件。和前面的区别是：
> 1. `CausalSelfAttention`：自动包含因果掩码（下三角矩阵），不需要手动传 mask
> 2. `MLP`：就是 FFN，名字换了（GPT 代码风格，和 Karpathy 的 nanoGPT 一致）
> 3. `Block`：就是 TransformerBlock，用了 config 字典来传参
>
> 代码结构和前面完全一样，就是工程上更规范了。不需要细讲，让学生自己看一下。
>
> 【运行 Cell 41-42】

## 📍 Cell 43（代码：GPT 类 — 完整定义）

⏱ **时间**：2:12-2:18（6分钟重点讲）

🎯 **核心概念**：GPT 完整实现 — wte, wpe, h, ln_f, lm_head, 权重共享

🗣 **讲课话术**：

> 这是整个下午最重要的一个 cell——完整的 GPT 模型。让我逐行讲：
>
> **`__init__` 部分**：
> ```python
> self.transformer = nn.ModuleDict(dict(
>     wte = nn.Embedding(vocab_size, n_embd),    # 词嵌入
>     wpe = nn.Embedding(block_size, n_embd),    # 位置嵌入
>     drop = nn.Dropout(dropout),                 # Dropout
>     h = nn.ModuleList([Block(config) for _ in range(n_layer)]),  # N个Block
>     ln_f = nn.LayerNorm(n_embd),               # 最终LayerNorm
> ))
> self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)  # 语言模型头
> ```
>
> 然后是权重共享那一行：
> ```python
> self.transformer.wte.weight = self.lm_head.weight
> ```
> 这不是复制，是让两个模块指向同一块内存。改一个，另一个也跟着变。
>
> **`forward` 部分**：
> 1. 创建位置序列 `pos = [0, 1, 2, ..., T-1]`
> 2. token embedding + position embedding → 加在一起
> 3. 过 dropout
> 4. 依次过所有 block
> 5. 最终 LayerNorm
> 6. lm_head 得到 logits
> 7. 如果有 targets，算 cross_entropy loss
>
> 【运行 Cell 43】

👀 **指出输出**：
- `GPT 参数量: 937,728` — 不到 100 万，这是一个迷你 GPT
- config: vocab=1000, block_size=128, 4层, 4头, 128维

🗣 **对比真实模型**：

> 我们这个小 GPT 不到 100 万参数。GPT-2 是 1.5 亿，GPT-3 是 1750 亿，GPT-4 据说是 1.7 万亿（MoE 架构）。结构完全一样，只是规模不同。这就是 scaling law 的精髓——架构不变，加大规模，能力涌现。

## 📍 Cell 44（代码：测试前向传播 + Loss）

⏱ **时间**：2:18-2:22（4分钟）

🎯 **核心概念**：验证 GPT 模型能跑，理解 logits 和 loss

🗣 **讲课话术**：

> 好，模型定义完了，来测试一下能不能跑。
>
> 输入：2 个样本，每个 20 个 token（随机整数，0-999）
>
> 【运行 Cell 44】
>
> 看输出：
> - 输入 `[2, 20]`：2 个样本，20 个 token ID
> - 输出 logits `[2, 20, 1000]`：每个位置都输出 1000 维的向量（对应 vocab_size=1000）
> - 每个位置的 1000 维向量就是"下一个词是什么"的原始分数
>
> 再看 Loss：**6.9751**。期望值是 ln(1000) ≈ 6.91。
>
> 为什么接近 ln(1000)？因为模型是随机初始化的，对 1000 个词的预测概率接近均匀分布（每个 1/1000）。交叉熵 = -log(1/1000) = log(1000) ≈ 6.91。
>
> 这是一个很好的 sanity check——如果初始 loss 和 ln(vocab_size) 差太多，说明模型实现有 bug。记住这个技巧！

❓ **常见问题**：
- **Q: logits 是什么？**
  - A: 未经 softmax 的原始分数。对 logits 做 softmax 就得到概率分布。名字来自 logistic function。
- **Q: cross_entropy loss 包含 softmax 吗？**
  - A: 是的！PyTorch 的 `F.cross_entropy` 内部会自动做 log_softmax，所以你传 logits 进去就行，不需要先做 softmax。

## 📍 Cell 45-46（代码：generate 函数 + 采样策略演示）

⏱ **时间**：2:22-2:30（8分钟）

🎯 **核心概念**：文本生成的采样策略 — Temperature, Top-k, Top-p

🗣 **讲课话术**：

> 模型有了，怎么用它生成文本？
>
> 最简单的方式——**自回归生成**（autoregressive generation）：
> 1. 给模型一些起始 token
> 2. 模型预测下一个 token
> 3. 把预测的 token 加到输入里
> 4. 重复步骤 2-3
>
> 但"预测下一个 token"有个问题——你用概率最高的那个（贪心）？还是按概率分布随机采样？
>
> 三种策略：
>
> **1. Temperature（温度）**
> - logits / temperature 再做 softmax
> - temperature < 1 → 分布更尖锐 → 更确定（倾向选概率最高的）
> - temperature > 1 → 分布更平坦 → 更随机（各选项概率更接近）
> - 类比：考试时 temperature 低就是"只答最有把握的"，temperature 高就是"啥都敢猜"
>
> **2. Top-k**
> - 只保留概率最高的 k 个 token，其他设为 0
> - k=1 就是贪心解码，k=50 是比较常用的值
> - 类比：只从成绩前 k 名的候选人里挑
>
> **3. Top-p（Nucleus Sampling）**
> - 保留累积概率达到 p 的最小 token 集合
> - p=0.9 意味着只从贡献了 90% 概率的那些 token 里采样
> - 优点：自适应——如果模型很确定，可能只保留 2-3 个；如果模型不确定，可能保留几十个
>
> 【运行 Cell 45（定义 generate 函数）和 Cell 46（演示对比）】

👀 **指出输出中的关键对比**：
- **Greedy (temp=0.1)**：看到有重复的 token——这是贪心的典型问题
- **Random (temp=1.5)**：每个 token 都不同，非常随机
- **Top-k (k=5)**：有些重复但比 greedy 多样一些
- **Top-p (p=0.9)**：比较好的平衡

> 注意：模型没训练过，输出是随机 token ID，看不出语义。但重点看**模式差异**——低 temperature 导致重复，高 temperature 导致混乱。

🗣 **企业实践建议**：

> 实际工作中怎么调这些参数？
> - **客服场景**：temperature 低（0.1-0.3），需要准确一致的回答
> - **创意写作**：temperature 高（0.7-1.0），需要多样性
> - **代码生成**：temperature 低（0.2-0.4），代码要精确
> - **通用对话**：temperature 中等（0.5-0.7），平衡创意和准确
>
> OpenAI API 和 Claude API 都支持这些参数，现在你知道它们背后的原理了！

## 📍 Cell 47-48（练习 4：自己实现 Top-k 采样）

⏱ **时间**：2:30-2:40（10分钟练习）

🎯 **核心概念**：从零实现采样逻辑，理解 temperature 和 top-k 的作用

🗣 **练习引导**：

> 好，最后两个练习。先做练习 4——自己实现 top-k 采样的两个核心函数：
>
> 1. `my_temperature_scale(logits, temperature)`：就是 logits / temperature，一行代码
> 2. `my_top_k_filter(logits, k)`：
>    - 用 `torch.topk` 找到前 k 大的值
>    - 用 `v[:, [-1]]` 得到第 k 大的值（阈值）
>    - 把小于阈值的 logit 设为 `-inf`
>
> 然后用三种不同的 (temperature, k) 配置对比概率分布，看柱状图。
>
> 5 分钟，试一下。

👀 **预期图形输出**：
- 三组柱状图，显示不同 (temperature, k) 下的概率分布
- 低 temperature + 小 k：柱子很少，最高的柱子接近 1
- 高 temperature + 大 k：柱子很多，高度接近均匀

🗣 **讲解要点**：

> 看图！temperature 控制"尖锐程度"——低温度下概率集中在少数 token 上，高温度下概率更分散。top-k 控制"候选范围"——k 越小，能被选中的 token 越少。
>
> 两者结合使用效果最好。在 ChatGPT 的 API 里，默认参数大约是 temperature=0.7, top_p=0.95。

---

# Mini-project + 练习 5（30 min）

## 📍 Cell 49-50（Mini-project：企业场景选生成策略）

⏱ **时间**：2:40-2:55（15分钟）

🎯 **核心概念**：把理论应用到企业实际场景

🗣 **练习引导**：

> 这个 mini-project 很贴近工作。三个场景：
> 1. **客服 Bot** — 要求准确一致
> 2. **营销文案** — 要求有创意
> 3. **代码补全** — 要求精确
>
> 你需要为每个场景选择合适的 temperature 和 top_k，写出理由，然后对比"好参数"和"差参数"的效果。
>
> 参考范围：
> - 客服：temperature 0.3-0.5，top_k 5-10
> - 营销：temperature 0.8-1.2，top_k 30-50
> - 代码：temperature 0.2-0.4，top_k 5-10
>
> 运行之后你会看到——用好参数生成 3 次，结果很一致或很有多样性；用差参数生成，客服回答不一致、创意文案很无聊、代码太随机。

👀 **指出输出中的关键对比**：
- 客服 Bot 好参数（temp=0.3, k=5）：3 次生成结果很接近，有重复 token（一致性好）
- 客服 Bot 差参数（temp=1.5, k=50）：3 次完全不同（不可靠）
- 营销文案好参数（temp=1.0, k=40）：3 次都不同（有创意）
- 营销文案差参数（temp=0.1, k=2）：大量重复 token（无聊死了）

🗣 **总结**：

> 这就是为什么调参很重要。同一个模型，参数选得好，是得力助手；参数选不好，是废物。工程师要根据业务需求选择合适的生成策略。

## 📍 Cell 51-52（练习 5：参数量估算器）

⏱ **时间**：2:55-3:10（15分钟）

🎯 **核心概念**：不创建模型，纯用公式估算参数量

🗣 **练习引导**：

> 最后一个练习——参数量估算。面试最爱考的题之一："GPT-3 有多少参数？怎么算的？"
>
> 公式我给你们了：
> - Token Embedding: `vocab_size * n_embd`
> - Position Embedding: `block_size * n_embd`
> - 每层 Attention (Q/K/V/O): `4 * n_embd^2 + 4 * n_embd`（4个线性层，每个 n_embd * n_embd 权重 + n_embd 偏置）
> - 每层 FFN: `8 * n_embd^2 + 5 * n_embd`（两个线性层，中间 4x 扩展。第一层 n_embd * 4n_embd + 4n_embd，第二层 4n_embd * n_embd + n_embd）
> - 每层 LayerNorm: `4 * n_embd`（两个 LN，每个有 gamma 和 beta 各 n_embd 个参数）
> - 最终 LN: `2 * n_embd`
> - LM Head: 与 Token Embedding 共享，不额外计数
>
> 实现 `estimate_params()` 函数，然后和前面创建的 GPT 对比验证。
>
> 最后，用这个公式预测 LLaMA-7B 的参数量！

**预期输出**：
```
各组件参数量:
  token_emb: 192,384
  pos_emb: 8,192
  per_layer_attn: 66,048
  per_layer_ffn: 131,712
  per_layer_ln: 512
  final_ln: 256
  total: 993,920

预测 LLaMA-7B 规模:
  估算参数量: 6,583,623,680 (6.58B)
```

🗣 **讲解要点**：

> 看 LLaMA-7B 的估算——6.58B。实际的 LLaMA-7B 是 6.7B 左右。差距是因为 LLaMA 用了 RMSNorm 代替 LayerNorm、SwiGLU 代替 GELU（FFN 结构略有不同），但数量级是对的！
>
> 再看各组件的占比：
> - per_layer_ffn (131,712) 是 per_layer_attn (66,048) 的 2 倍
> - FFN 真的是参数大户！
> - token_emb 在小模型里占比很大（vocab_size * n_embd），但在大模型里占比很小
>
> 这个公式面试的时候能现推就加分。关键记住：每层大约 `12 * n_embd^2` 参数（4 for attn + 8 for ffn）。

❓ **常见问题**：
- **Q: 为什么 Attention 是 4 * n_embd^2？**
  - A: Q/K/V/O 四个线性层，每个都是 [n_embd, n_embd]，所以 4 * n_embd^2。多头不增加参数量——拆分只是 reshape，不增加新的参数。
- **Q: 权重共享省了多少参数？**
  - A: vocab_size * n_embd。对于 LLaMA-7B (vocab=32000, n_embd=4096)，省了 32000*4096 = 1.3 亿参数。不少！

---

# Part 6 | 架构对比 + 总结（20 min）

## 📍 Cell 53（Markdown — GPT vs BERT vs T5 对比表）

⏱ **时间**：3:10-3:20（10分钟）

🎯 **核心概念**：三大 Transformer 流派的区别和选型建议

🗣 **讲课话术**：

> 最后，让我们把视角拉高一点。Transformer 架构催生了三大流派，各有各的擅长：
>
> **1. GPT（Decoder-only）** — 我们今天搭的就是这个
> - 代表：GPT-4、Claude、LLaMA
> - 特点：因果掩码，单向注意力，预测下一个词
> - 擅长：文本生成、对话、推理
>
> **2. BERT（Encoder-only）** — 双向注意力
> - 代表：BERT、RoBERTa
> - 特点：没有因果掩码，能看到全文
> - 擅长：文本分类、情感分析、NER
> - 训练方式：随机遮住 15% 的词，让模型猜（MLM, Masked Language Modeling）
>
> **3. T5（Encoder-Decoder）** — 两段式
> - 代表：T5、mT5、BART
> - 特点：编码器双向 + 解码器单向
> - 擅长：翻译、摘要、问答
>
> **怎么选？简单规则**：
> - 要**生成**文本 → GPT
> - 要**理解/分类** → BERT（轻量级任务，成本低）
> - 要**转换**（翻译、摘要） → T5
> - **什么都要** → 大 GPT（能力涌现）
>
> 2024-2025 的趋势：Decoder-only 一统天下。为什么？因为 GPT-4、Claude 这种足够大的 decoder-only 模型，理解和生成都能做好。BERT 还在企业里用，但主要是因为便宜和快。

👀 **指出表格中的关键对比**：
- 注意力方向：单向 vs 双向 vs 混合
- 训练目标：预测下一个词 vs 填空 vs Seq2Seq
- 企业用途一列特别重要——和工作直接相关

❓ **常见问题**：
- **Q: 既然 GPT 什么都能做，BERT 还有存在的必要吗？**
  - A: 有！BERT-base 只有 1.1 亿参数，推理速度快、成本低。如果你只需要做文本分类（比如垃圾邮件检测），用 GPT-4 杀鸡用牛刀，BERT 又快又便宜。
- **Q: T5 还有人用吗？**
  - A: Google 的 Flan-T5 仍然很流行，特别是需要结构化输出（如翻译、提取信息）的场景。但新的研究趋势确实在向 decoder-only 集中。

## 📍 Cell 54（Markdown — Day 1 总结 + Day 2 预告）

⏱ **时间**：3:20-3:30（10分钟总结）

🎯 **核心概念**：回顾一天所学，展望明天

🗣 **讲课话术**：

> 好，今天一整天的内容回顾一下：
>
> **上午**：我们搞懂了深度学习的基础——前向传播、反向传播、梯度下降，然后学了词嵌入，把离散的词变成连续的向量。
>
> **下午**：我们从"为什么静态向量不够"出发，学了 Self-Attention 的 QKV 机制，然后是 Multi-Head Attention 的多头分工。休息回来搭了 Transformer Block（残差+LN+FFN），最后组装了一个完整的 GPT 模型，还学了文本生成的采样策略。
>
> 一天下来，你们已经亲手搭了一个 GPT！虽然只有不到 100 万参数，但结构和 GPT-4 是一样的。
>
> 两个最重要的公式：
>
> $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$
>
> $$\text{TransformerBlock}(x) = x + \text{FFN}(\text{LN}(x + \text{Attention}(\text{LN}(x))))$$
>
> 记住这两个，面试的时候能说出来就过了。
>
> **明天预告**：
> - **上午**：SFT + LoRA 微调实战 — 怎么用自己的数据微调大模型，而且只改 1% 的参数
> - **下午**：预训练 + 对齐优化 — 大模型是怎么训练出来的，RLHF/DPO 怎么让模型对齐人类偏好
>
> 明天的内容更贴近实际工作，会很有意思。好好休息，明天见！

**讲师注意事项**：
- 提醒学生保存 notebook
- 如果有未完成的练习，鼓励晚上自行完成
- 把练习的 hints 指给还在纠结的学生看

---

# 附录：Cell 编号与 Notebook 对照表

| Cell # | 类型 | 内容摘要 | 所属 Part |
|--------|------|---------|----------|
| 1 | Code | sys.path 配置 | Setup |
| 2 | Code | 中文字体配置 | Setup |
| 3 | Code | PyTorch import | Setup |
| 4 | Markdown | Part 1 标题 + 静态向量问题 | Part 1 |
| 5 | Markdown | QKV 直觉理解（图书馆比喻） | Part 1 |
| 6 | Markdown | Part 2 标题 + 核心公式 | Part 2 |
| 7 | Markdown | Step-by-step 标题 | Part 2 |
| 8 | Code | 创建输入 x（猫坐垫，3词4维） | Part 2 |
| 9 | Code | 线性投影得到 Q, K, V | Part 2 |
| 10 | Code | 计算 scores = QK^T / sqrt(d_k) | Part 2 |
| 11 | Code | Softmax 得到 attention_weights | Part 2 |
| 12 | Code | 加权求和得到输出 | Part 2 |
| 13 | Code | 注意力权重热力图 | Part 2 |
| 14 | Markdown | 完整 Self-Attention 函数标题 | Part 2 |
| 15 | Code | self_attention() 函数 + 测试 | Part 2 |
| 16-17 | Code | 因果掩码可视化 | Part 2 |
| 18 | Markdown | 练习 1 说明 | Part 2 |
| 19 | Code | 练习 1：手算 Attention 分数 | Part 2 |
| 20 | Markdown | Part 3 标题 + Multi-Head 说明 | Part 3 |
| 21 | Code | MultiHeadAttention 类 | Part 3 |
| 22 | Code | 不同 Head 注意力模式可视化 | Part 3 |
| 23 | Markdown | 练习 2 说明 | Part 3 |
| 24 | Code | 练习 2：注意力模式侦探 | Part 3 |
| 25 | Markdown | 休息 10 分钟 | Break |
| 26 | Markdown | Part 4 标题 + 架构图 | Part 4 |
| 27 | Markdown | 残差连接说明 | Part 4 |
| 28 | Code | 残差连接对比实验 | Part 4 |
| 29 | Markdown | LayerNorm 公式 | Part 4 |
| 30 | Code | 手写 LayerNorm + 对比 PyTorch | Part 4 |
| 31 | Markdown | FFN 说明 | Part 4 |
| 32 | Code | FeedForward 类 | Part 4 |
| 33 | Markdown | 组装标题 | Part 4 |
| 34 | Code | MultiHeadAttentionV2 类 | Part 4 |
| 35 | Code | TransformerBlock 类 | Part 4 |
| 36 | Markdown | 练习 3 说明 | Part 4 |
| 37 | Code | 练习 3：拼装 TransformerBlock | Part 4 |
| 38 | Code | 堆叠 Transformer (6层) | Part 5 |
| 39-40 | Markdown | GPT 完整结构 + 位置编码 | Part 5 |
| 41-42 | Code | CausalSelfAttention + Block | Part 5 |
| 43 | Code | GPT 类完整定义 | Part 5 |
| 44 | Code | 测试前向传播 + Loss | Part 5 |
| 45 | Code | generate 函数定义 | Part 5 |
| 46 | Code | 采样策略对比演示 | Part 5 |
| 47 | Markdown | 练习 4 说明 | Part 5 |
| 48 | Code | 练习 4：Top-k 采样实现 | Part 5 |
| 49 | Markdown | Mini-project 说明 | Mini-project |
| 50 | Code | Mini-project：企业场景 | Mini-project |
| 51 | Markdown | 练习 5 说明 | 练习 5 |
| 52 | Code | 练习 5：参数量估算器 | 练习 5 |
| 53 | Markdown | GPT vs BERT vs T5 对比 | Part 6 |
| 54 | Markdown | Day 1 总结 + Day 2 预告 | 总结 |

---

# 附录：讲师备忘清单

## 上课前
- [ ] 运行所有 cell 确保无报错（特别注意字体和 PyTorch 版本）
- [ ] 准备白板/画板用于画架构图
- [ ] 准备计时器控制练习时间
- [ ] 确认学生环境已配置好（上午应该已经搞定）

## 节奏控制
- Part 1 (10min): 不要超时，这是铺垫
- Part 2 (40min): 最重要的部分，可以适当超时 5 分钟
- Part 3 (25min): 练习 2 如果学生卡壳，直接给答案讲解
- 休息: 严格 10 分钟
- Part 4 (35min): FFN 和 LayerNorm 可以快速过
- Part 5 (40min): GPT 类定义是重点，采样策略可以快讲
- 练习 5 + Mini-project (30min): 如果时间不够，练习 5 可以留作课后
- Part 6 (20min): 对比表可以快速过，重点放在总结和预告

## 关键类比（随时可以用）
- QKV = 图书馆找书（Q=搜索词，K=书标签，V=书内容）
- 残差连接 = 高速公路/电梯直达
- LayerNorm = 稳压器
- FFN = 加工厂（先展开再压缩）
- Multi-Head = 多个专家同时分析
- Temperature = 考试时的"冒险程度"
- Transformer Block = 完整的神经系统

## 常见翻车场景
- 热力图中文乱码 → 重新运行字体配置 cell
- 练习时间过长 → 提前给 hints，必要时直接展示答案
- 学生问 "这和 ChatGPT 的区别" → 回答：结构一样，我们这个是 mini 版，差别在规模和训练数据
- 学生问 "为什么 loss 不是 0" → 回答：模型没训练过，随机猜的，训练之后 loss 会下降